# 🛍️ Customer Segmentation Analysis
### End-to-End Notebook — Python · ML · Business Intelligence

**Goal:** segment customers into actionable personas using K-Means, RFM, PCA and 20+ visualizations.

**Workflow:**
1. Load data  
2. Clean data  
3. Feature engineering  
4. EDA (22 visualizations)  
5. Preprocessing (encoding + scaling)  
6. K-Means + Elbow + Silhouette  
7. PCA visualization  
8. Cluster profiles & personas  
9. Business insights

> All heavy lifting lives in the `src/` package — this notebook is the narrative.

## 1 · Imports & Setup

In [ ]:
import sys, os
from pathlib import Path
# Make `src` importable when running from /notebooks
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_dataset
from src.data_cleaning import clean_data
from src.feature_engineering import engineer_features
from src.eda import run_eda
from src.preprocessing import build_feature_matrix
from src.clustering import (
    elbow_and_silhouette, fit_kmeans, visualize_pca,
    cluster_profile_heatmap, cluster_size_donut, cluster_revenue_bar,
)
from src.visualization import set_style
set_style()
print('Setup complete.')

## 2 · Load Data
Auto-downloads the Customer Personality Analysis dataset; falls back to a high-fidelity synthetic generator if offline.

In [ ]:
df_raw = load_dataset()
print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

## 3 · Data Cleaning
- Drop constant columns
- Fix dtypes (dates)
- Remove duplicates
- Impute missing values
- Consolidate noisy categoricals
- Cap outliers (IQR)

In [ ]:
df_clean = clean_data(df_raw)
df_clean.head()

## 4 · Feature Engineering
Adds Age, Tenure, Family Size, Income Group, Total/Average Spend, Loyalty Score, and RFM features.

In [ ]:
df_feat = engineer_features(df_clean)
df_feat[['Age','Income_Group','Total_Spend','Total_Purchases','Loyalty_Score',
         'Recency','Frequency','Monetary']].describe().round(2)

## 5 · Exploratory Data Analysis (22 visualizations)
`run_eda` writes all charts to `/visualizations`.

In [ ]:
paths = run_eda(df_feat)
print(f'Saved {len(paths)} charts.')

**Key visual insights**
- Income is right-skewed; a premium tier drives outsized revenue.
- Wines & Meat dominate product spend.
- Store > Web > Catalog > Deals as channels.
- Strong Income ↔ Total Spend correlation; parents underspend at any given income.
- Campaign acceptance ≤ 10% historically → personalization is needed.

## 6 · Preprocessing
Ordinal-encode Education, one-hot encode Marital Status, then `StandardScaler` for all numeric features (essential for K-Means).

In [ ]:
X_scaled, scaler = build_feature_matrix(df_feat)
X_scaled.head()

## 7 · K-Means — Elbow + Silhouette
We use the **elbow** (WCSS) to spot diminishing returns and the **silhouette score** to quantitatively pick `k`.

In [ ]:
diag = elbow_and_silhouette(X_scaled, k_range=range(2, 9))
best_k_pool = [k for k in diag['k'] if 3 <= k <= 6]
best_scores = [diag['silhouette'][diag['k'].index(k)] for k in best_k_pool]
k_opt = best_k_pool[int(np.argmax(best_scores))]
print('Optimal k =', k_opt)

In [ ]:
km = fit_kmeans(X_scaled, k=k_opt)
df_feat['Cluster'] = km.labels_
df_feat['Cluster'].value_counts().sort_index()

## 8 · PCA — 2-D Cluster Visualization

In [ ]:
visualize_pca(X_scaled, km.labels_, k=k_opt)

## 9 · Cluster Profiles

In [ ]:
profile_cols = ['Income','Age','Total_Spend','Total_Purchases','Recency',
                'Loyalty_Score','Children','Customer_Tenure_Days','Total_Accepted_Cmp']
profile = df_feat.groupby('Cluster')[profile_cols].mean().round(2)
profile

In [ ]:
cluster_profile_heatmap(profile)
cluster_size_donut(km.labels_)
cluster_revenue_bar(df_feat)

## 10 · Personas & Business Recommendations

| Persona | Profile | Strategy |
|---------|---------|----------|
| 🟢 **Premium Loyalists** | High income, high spend, frequent, low recency | VIP program, premium upsell, referral rewards |
| 🔵 **Regular Customers** | Mid income, steady spend & frequency | Loyalty points, bundles, free-shipping thresholds |
| 🟡 **High Potential** | High income, low engagement | Re-engagement journeys, personalized offers |
| 🔴 **Budget / At-Risk** | Low income, high recency, low spend | Win-back discount, churn prevention |

Detailed action plan: see `reports/final_insights.md`.

---
### ✅ Pipeline complete
Outputs:  `visualizations/` · `models/` · `data/processed/` · `reports/`